# Vulnerability indicator leave-one-out scenarios

This notebook loads all leave-one-out scenario raster layers of the lack of adaptive capacity dimension and the sensitivity dimension. It aggregates all scenario rasters from one dimension once together with the baseline scenario (all variables included) from the other dimension, leading to ten scenarios of overall vulnerability. It aggregates through simple additive aggregation.  It than creates: 
- a raster with mean vulnerability scores at a chosen pixel size for ten different scenarios
- a raster with vulnerability score at a chosen pixel size (not normalized) for ten different scenarios

## How to run
1. Put the required input files in the same folder as this notebook (or edit the paths in the **Configuration** cell below).
2. Run the cells from top to bottom.

## Required files
- `sensitivity_mean.tif`
- `lackof_adapt_mean.tif`
- `sensitivity_mean_omit_*.tif` (six scenarios)
- `lackof_adapt_mean_omit_*.tif` (four scenarios)
- `bii_5000m.tif`
- `World_Countries_(Generalized)_8414823838130214587.gpkg` (or your country layer)

In [ ]:
#import packages
import pandas as pd
import geopandas as gpd
import os, math, glob
import numpy as np
import rasterio
from rasterio.windows import Window
from rasterio.enums import Resampling
from rasterio import warp
from rasterio import features
import geopandas as gpd
from shapely.geometry import box
from rasterio.features import geometry_mask
from shapely.ops import unary_union
from rasterio.features import rasterize
import matplotlib.ticker as mticker
from matplotlib.lines import Line2D
from shapely.geometry import box
from pathlib import Path

In [ ]:
# Configuration (edit these)

BII_5000M =  'biodiversity_intactness\\bii_5000m.tif'  
WORLD_COUNTRIES_GENERAL = 'lackof_adapt\\World_Countries_(Generalized)_8414823838130214587.gpkg' 

# Complete dimensions
BASE_SENS  = 'sensitivity\\sensitivity_mean.tif'
BASE_ADAPT ='lackof_adapt\\lackof_adapt_mean.tif'  

# Leave-one-out scenario rasters (means)
SENS_PATTERN  = 'sensitivity\\scenarios\\scenarios\\sensitivity_mean_omit_*.tif'
ADAPT_PATTERN = 'lackof_adapt\\scenarios\\scenarios\\lackof_adapt_mean_omit_*.tif'

#other parameters
window_size = 2048
dst_nodata = -9999.0

# Output
OUT_DIR ='vulnerability_indicator\\scenarios_missing_variable\\scenarios_one_missing'
os.makedirs(OUT_DIR, exist_ok=True)

out_mask = os.path.join(OUT_DIR, "country_mask_vulnerability.tif")

COUNTRY_ID = 'vulnerability_indicator\\country_id.tif'  
LOOKUP = 'vulnerability_indicator\\country_id_lookup.csv' 

In [ ]:
# Inputs
ref_path = BII_5000M
countries_path = WORLD_COUNTRIES_GENERAL


# Load reference grid
with rasterio.open(ref_path) as ref:
    ref_meta = ref.meta.copy()
    ref_crs = ref.crs
    ref_transform = ref.transform
    ref_width = ref.width
    ref_height = ref.height


# Country mask
def build_country_mask():
    if os.path.exists(out_mask):
        print(f"country mask exists, skipping: {out_mask}")
        return

    gdf = gpd.read_file(countries_path).to_crs(ref_crs)

    gdf = gdf[gdf["COUNTRY"] != "Antarctica"].copy()
    gdf = gdf[gdf.geometry.notnull()].copy()
    gdf["geometry"] = gdf.geometry.buffer(0)
    gdf = gdf[gdf.is_valid].copy()
    gdf = gdf[gdf.geometry.notnull()].copy()

    sindex = gdf.sindex

    mask_meta = ref_meta.copy()
    mask_meta.update(
        dtype="uint8", count=1, nodata=0,
        compress="deflate", predictor=2, tiled=True,
        blockxsize=256, blockysize=256
    )

    with rasterio.open(out_mask, "w", **mask_meta) as dst:
        n_rows = math.ceil(ref_height / window_size)
        n_cols = math.ceil(ref_width / window_size)

        for row in range(n_rows):
            for col in range(n_cols):
                x_off = col * window_size
                y_off = row * window_size
                w = min(window_size, ref_width - x_off)
                h = min(window_size, ref_height - y_off)
                window = Window(x_off, y_off, w, h)

                win_bounds = rasterio.windows.bounds(window, ref_transform)
                win_geom = box(*win_bounds)

                px = max(abs(ref_transform.a), abs(ref_transform.e))
                win_geom = win_geom.buffer(px)

                cand_idx = list(sindex.intersection(win_geom.bounds))
                if not cand_idx:
                    dst.write(np.zeros((h, w), dtype=np.uint8), 1, window=window)
                    continue

                sub = gdf.iloc[cand_idx]
                sub = sub[sub.intersects(win_geom)]
                if sub.empty:
                    dst.write(np.zeros((h, w), dtype=np.uint8), 1, window=window)
                    continue

                win_transform = rasterio.windows.transform(window, ref_transform)
                shapes = ((geom, 1) for geom in sub.geometry)
                burned = features.rasterize(
                    shapes=shapes,
                    out_shape=(h, w),
                    transform=win_transform,
                    fill=0,
                    all_touched=False,
                    dtype="uint8",
                )
                dst.write(burned, 1, window=window)

    print(f"country mask: {out_mask}")


def stem(p):
    return os.path.splitext(os.path.basename(p))[0]


def compute_vulnerability_pair(sens_path, adapt_path, scenario_tag, out_dir=OUT_DIR):
    # consistent names
    out_sum  = os.path.join(out_dir, f"vulnerability__{scenario_tag}.tif")
    out_mean = os.path.join(out_dir, f"vulnerability_mean__{scenario_tag}.tif")

    sum_meta = ref_meta.copy()
    sum_meta.update(
        dtype="float32", count=1, nodata=dst_nodata,
        compress="deflate", predictor=2, tiled=True, blockxsize=256, blockysize=256
    )
    mean_meta = sum_meta.copy()

    src_s = rasterio.open(sens_path)
    src_a = rasterio.open(adapt_path)
    msk = rasterio.open(out_mask)

    try:
        with rasterio.open(out_sum, "w", **sum_meta) as dst_sum, \
             rasterio.open(out_mean, "w", **mean_meta) as dst_mean:

            n_rows = math.ceil(ref_height / window_size)
            n_cols = math.ceil(ref_width / window_size)

            for row in range(n_rows):
                for col in range(n_cols):
                    x_off = col * window_size
                    y_off = row * window_size
                    w = min(window_size, ref_width - x_off)
                    h = min(window_size, ref_height - y_off)
                    window = Window(x_off, y_off, w, h)

                    country = msk.read(1, window=window).astype(bool)

                    s_arr = src_s.read(1, window=window)
                    a_arr = src_a.read(1, window=window)

                    s_valid = np.isfinite(s_arr) & (s_arr != dst_nodata)
                    a_valid = np.isfinite(a_arr) & (a_arr != dst_nodata)

                    use_s = country & s_valid
                    use_a = country & a_valid

                    sum_arr = np.zeros((h, w), dtype=np.float32)
                    known = np.zeros((h, w), dtype=np.uint8)

                    if np.any(use_s):
                        sum_arr[use_s] += s_arr[use_s].astype(np.float32)
                        known[use_s] += 1
                    if np.any(use_a):
                        sum_arr[use_a] += a_arr[use_a].astype(np.float32)
                        known[use_a] += 1

                    has_data = country & (known > 0)

                    out_sum_arr = np.full((h, w), dst_nodata, dtype=np.float32)
                    out_mean_arr = np.full((h, w), dst_nodata, dtype=np.float32)

                    out_sum_arr[has_data] = sum_arr[has_data]
                    out_mean_arr[has_data] = sum_arr[has_data] / known[has_data].astype(np.float32)

                    dst_sum.write(out_sum_arr, 1, window=window)
                    dst_mean.write(out_mean_arr, 1, window=window)

    finally:
        src_s.close()
        src_a.close()
        msk.close()

    print("Wrote:", out_mean)


def run_one_missing_scenarios(write_baseline=True):
    build_country_mask()

    # Baseline-baseline 
    if write_baseline:
        compute_vulnerability_pair(
            BASE_SENS, BASE_ADAPT,
            scenario_tag="S_all__A_all"
        )

    # 1) Sensitivity leave-one-out + Adapt baseline
    sens_scen = sorted(glob.glob(SENS_PATTERN))
    if not sens_scen:
        print("WARNING: no sensitivity rasters found:", SENS_PATTERN)
    for s_path in sens_scen:
        omit = stem(s_path).replace("sensitivity_mean_omit_", "")  
        tag = f"S_omit_{omit}__A_all"
        compute_vulnerability_pair(s_path, BASE_ADAPT, scenario_tag=tag)

    # 2) Adapt leave-one-out + Sens baseline
    adapt_scen = sorted(glob.glob(ADAPT_PATTERN))
    if not adapt_scen:
        print("WARNING: no adapt rasters found:", ADAPT_PATTERN)
    for a_path in adapt_scen:
        omit = stem(a_path).replace("lackof_adapt_mean_omit_", "")  
        tag = f"S_all__A_omit_{omit}"
        compute_vulnerability_pair(BASE_SENS, a_path, scenario_tag=tag)


# Execute
run_one_missing_scenarios(write_baseline=True)


In [1]:
# load normalized variables used to build sensitivity_mean and lackof_adapt_mean
sens_vars = {
    "Biodiversity Intactness":    'C:\\Users\\janab\\Documents\\Jupyter\\sensitivity\\biodiversity_intactness\\bii_5000m.tif',
    "Protected Areas":  'C:\\Users\\janab\\Documents\\Jupyter\\sensitivity\\protected_areas\\wdpa_5000m.tif',
    "IPLC Lands": 'C:\\Users\\janab\\Documents\\Jupyter\\sensitivity\\ind_com_lands\\landmark_5000m.tif' ,
    "Key Biodiversity Areas":'C:\\Users\\janab\\Documents\\Jupyter\\sensitivity\\kba\\kba_5000m.tif',
    "Poverty":   'C:\\Users\\janab\\Documents\\Jupyter\\sensitivity\\poverty\\poverty_5000m.tif',
    "Water Risk": 'C:\\Users\\janab\\Documents\\Jupyter\\sensitivity\\water_risk\\water_risk_5000m.tif',
}

lackof_adapt_vars= {
    "Conflict":  'C:\\Users\\janab\\Documents\\Jupyter\\lackof_adapt\\conflict\\conflict_5000m.tif',
    "Environmental Democracy": 'C:\\Users\\janab\\Documents\\Jupyter\\lackof_adapt\\environmental_democracy\\edi_5000m.tif',
    "Landrights":'C:\\Users\\janab\\Documents\\Jupyter\\lackof_adapt\\landrights\\landrights_5000m.tif',
    "Rule of Law":   'C:\\Users\\janab\\Documents\\Jupyter\\lackof_adapt\\rule_of_law\\rule_of_law_5000m.tif'
}


In [ ]:
#Calculate variable contribtuion to mean vulnerability (different scenarios

country_id_path = COUNTRY_ID
lookup_csv      = LOOKUP

# Folder that contains all vulnerability_mean scenario rasters
vuln_scen_dir = OUT_DIR
vuln_mean_pattern = os.path.join(vuln_scen_dir, "vulnerability_mean__*.tif")


# Output folder for per-scenario CSVs
out_csv_dir = os.path.join(vuln_scen_dir, "country_means_scenarios")
os.makedirs(out_csv_dir, exist_ok=True)


# BASE VARIABLE RASTERS
sens_vars_all = {
    "bii":        "/home/ubuntu/venv/sensitivity/biodiversity_intactness/bii_5000m.tif",
    "wdpa":       "/home/ubuntu/venv/sensitivity/protected_areas/wdpa_5000m.tif",
    "landmark":   "/home/ubuntu/venv/sensitivity/ind_com_lands/landmark_5000m.tif",
    "kba":        "/home/ubuntu/venv/sensitivity/kba/kba_5000m.tif",
    "poverty":    "/home/ubuntu/venv/sensitivity/poverty/poverty_5000m.tif",
    "water_risk": "/home/ubuntu/venv/sensitivity/water_stress/water_risk_5000m.tif",
}

lack_vars_all = {
    "conflict":     "/home/ubuntu/venv/adaptive_capacity/conflict/conflict_5000m.tif",
    "edi":          "/home/ubuntu/venv/adaptive_capacity/environment_regulation/edi_5000m.tif",
    "landrights":   "/home/ubuntu/venv/adaptive_capacity/land_rights/landrights_5000m.tif",
    "rule_of_law":  "/home/ubuntu/venv/adaptive_capacity/rule_of_law/rule_of_law_5000m.tif",
}


# Helper: map omitted filename stem -> variable key
def _stem(p):
    return os.path.splitext(os.path.basename(p))[0]

def find_key_by_stem(vars_dict, omitted_stem):
    """Return key in vars_dict whose path basename stem matches omitted_stem."""
    for k, p in vars_dict.items():
        if _stem(p) == omitted_stem:
            return k
    return None

def parse_omits_from_vuln_mean_filename(vuln_mean_path):
    """
    Expected filenames:
      vulnerability_mean__S_all__A_all.tif
      vulnerability_mean__S_omit_bii_5000m__A_all.tif
      vulnerability_mean__S_all__A_omit_conflict_5000m.tif

    Returns (sens_omit_stem, lack_omit_stem)
      e.g. ("bii_5000m", None) or (None, "conflict_5000m") or (None, None)
    """
    name = _stem(vuln_mean_path)

    prefix = "vulnerability_mean__"
    if name.startswith(prefix):
        name = name[len(prefix):]

    parts = name.split("__A_")
    if len(parts) != 2:
        return (None, None)

    s_part, a_part = parts[0], parts[1]

    # s_part looks like: "S_all" or "S_omit_bii_5000m"
    sens_omit_stem = None
    if s_part.startswith("S_omit_"):
        sens_omit_stem = s_part[len("S_omit_"):]

    # a_part looks like: "all" or "omit_conflict_5000m"
    lack_omit_stem = None
    if a_part.startswith("all"):
        lack_omit_stem = None
    elif a_part.startswith("omit_"):
        lack_omit_stem = a_part[len("omit_"):]
    elif a_part.startswith("A_omit_"):
        # (just in case you ever include A_ in that part)
        lack_omit_stem = a_part[len("A_omit_"):]

    return (sens_omit_stem, lack_omit_stem)


#function to calculate contribution
def compute_country_variable_contrib(country_id_path, lookup_csv, vuln_mean_path, out_csv,
                                     sens_vars, lackof_adapt_vars):
    lookup = pd.read_csv(lookup_csv)
    max_id = int(lookup["country_id"].max())

    count = np.zeros(max_id + 1, dtype=np.int64)
    sum_v = np.zeros(max_id + 1, dtype=np.float64)

    sum_contrib = {f"sens_{k}": np.zeros(max_id + 1, dtype=np.float64) for k in sens_vars}
    sum_contrib.update({f"lack_{k}": np.zeros(max_id + 1, dtype=np.float64) for k in lackof_adapt_vars})

    sens_srcs = {k: rasterio.open(p) for k, p in sens_vars.items()}
    lack_srcs = {k: rasterio.open(p) for k, p in lackof_adapt_vars.items()}
    vsrc = rasterio.open(vuln_mean_path)
    csrc = rasterio.open(country_id_path)

    try:
        n_rows = math.ceil(vsrc.height / window_size)
        n_cols = math.ceil(vsrc.width / window_size)

        for row in range(n_rows):
            for col in range(n_cols):
                x_off = col * window_size
                y_off = row * window_size
                w = min(window_size, vsrc.width - x_off)
                h = min(window_size, vsrc.height - y_off)
                window = Window(x_off, y_off, w, h)

                cid = csrc.read(1, window=window)
                v = vsrc.read(1, window=window)

                valid_v = (cid > 0) & np.isfinite(v) & (v != dst_nodata)
                if not np.any(valid_v):
                    continue

                cc = cid[valid_v].astype(np.int32)
                vv = v[valid_v].astype(np.float32)

                count += np.bincount(cc, minlength=max_id + 1)
                sum_v += np.bincount(cc, weights=vv.astype(np.float64), minlength=max_id + 1)

               # sensitivity vars
                sens_stack = []
                sens_valid_stack = []
                for k, src in sens_srcs.items():
                    a = src.read(1, window=window)
                    a_valid = np.isfinite(a) & (a != dst_nodata)
                    sens_stack.append(a)
                    sens_valid_stack.append(a_valid)

                kS = np.zeros_like(v, dtype=np.float32)
                for a_valid in sens_valid_stack:
                    kS[valid_v] += a_valid[valid_v].astype(np.float32)
                kS[valid_v] = np.maximum(kS[valid_v], 1.0)

                for (k, a, a_valid) in zip(sens_vars.keys(), sens_stack, sens_valid_stack):
                    use = valid_v & a_valid
                    if np.any(use):
                        contrib = 0.5 * (a[use].astype(np.float32) / kS[use])
                        sum_contrib[f"sens_{k}"] += np.bincount(
                            cid[use].astype(np.int32),
                            weights=contrib.astype(np.float64),
                            minlength=max_id + 1
                        )

                #  lack of adapt vars
                lack_stack = []
                lack_valid_stack = []
                for k, src in lack_srcs.items():
                    a = src.read(1, window=window)
                    a_valid = np.isfinite(a) & (a != dst_nodata)
                    lack_stack.append(a)
                    lack_valid_stack.append(a_valid)

                kL = np.zeros_like(v, dtype=np.float32)
                for a_valid in lack_valid_stack:
                    kL[valid_v] += a_valid[valid_v].astype(np.float32)
                kL[valid_v] = np.maximum(kL[valid_v], 1.0)

                for (k, a, a_valid) in zip(lackof_adapt_vars.keys(), lack_stack, lack_valid_stack):
                    use = valid_v & a_valid
                    if np.any(use):
                        contrib = 0.5 * (a[use].astype(np.float32) / kL[use])
                        sum_contrib[f"lack_{k}"] += np.bincount(
                            cid[use].astype(np.int32),
                            weights=contrib.astype(np.float64),
                            minlength=max_id + 1
                        )

        out = lookup.copy()
        out["pixel_count"] = out["country_id"].map(lambda i: int(count[int(i)]))
        out["mean_vulnerability"] = out["country_id"].map(
            lambda i: float(sum_v[int(i)] / count[int(i)]) if count[int(i)] > 0 else np.nan
        )

        for key in sum_contrib:
            out[f"contrib_{key}"] = out["country_id"].map(
                lambda i: float(sum_contrib[key][int(i)] / count[int(i)]) if count[int(i)] > 0 else np.nan
            )

        sens_cols = [f"contrib_sens_{k}" for k in sens_vars]
        lack_cols = [f"contrib_lack_{k}" for k in lackof_adapt_vars]

        out["contrib_sensitivity_total"] = out[sens_cols].sum(axis=1) if sens_cols else 0.0
        out["contrib_lack_total"] = out[lack_cols].sum(axis=1) if lack_cols else 0.0
        out["contrib_sum_vars"] = out["contrib_sensitivity_total"] + out["contrib_lack_total"]
        out["diff_vs_vuln"] = out["contrib_sum_vars"] - out["mean_vulnerability"]

        out.to_csv(out_csv, index=False)
        print(out_csv)

    finally:
        for src in sens_srcs.values():
            src.close()
        for src in lack_srcs.values():
            src.close()
        vsrc.close()
        csrc.close()



# Main loop: run for every vulnerability scenario
def run_country_means_for_all_vuln_scenarios():
    vuln_means = sorted(glob.glob(vuln_mean_pattern))
    if not vuln_means:
        raise RuntimeError(f"No vulnerability mean rasters found: {vuln_mean_pattern}")

    print(f"Found {len(vuln_means)} vulnerability scenarios")

    for vpath in vuln_means:
        sens_omit_stem, lack_omit_stem = parse_omits_from_vuln_mean_filename(vpath)

        # Determine omitted keys (by matching raster basename stems)
        sens_omit_key = find_key_by_stem(sens_vars_all, sens_omit_stem) if sens_omit_stem else None
        lack_omit_key = find_key_by_stem(lack_vars_all, lack_omit_stem) if lack_omit_stem else None

        # Build scenario dicts (remove omitted var if recognized)
        sens_vars = dict(sens_vars_all)
        lack_vars = dict(lack_vars_all)

        if sens_omit_key and sens_omit_key in sens_vars:
            sens_vars.pop(sens_omit_key)
        if lack_omit_key and lack_omit_key in lack_vars:
            lack_vars.pop(lack_omit_key)

        # Output CSV name matches scenario raster stem
        scenario_name = _stem(vpath).replace("vulnerability_mean__", "")
        out_csv = os.path.join(out_csv_dir, f"country_means_{scenario_name}.csv")

        print("\nScenario:", scenario_name)
        if sens_omit_stem:
            print("  sensitivity omitted stem:", sens_omit_stem, "-> key:", sens_omit_key)
        if lack_omit_stem:
            print("  lack-of-adapt omitted stem:", lack_omit_stem, "-> key:", lack_omit_key)

        compute_country_variable_contrib(
            country_id_path=country_id_path,
            lookup_csv=lookup_csv,
            vuln_mean_path=vpath,
            out_csv=out_csv,
            sens_vars=sens_vars,
            lackof_adapt_vars=lack_vars
        )

run_country_means_for_all_vuln_scenarios()